# Approach 2 — MultiTaskCNN Training

**Before running:** Runtime → Change runtime type → **T4 GPU**

### What this trains
`MultiTaskCNN`: shared CNN backbone with two heads —  
- **Character head**: predicts which letter/digit (a-z, A-Z, 0-9)  
- **Writer head**: predicts which of the 6 writers wrote it  

This is a **classifier, not a generator**. Output is a predictions grid showing real images with true vs predicted labels.

### After training
Download `checkpoint.pt` and place it at:  
`approaches/02_MultiTaskCNN/checkpoints/checkpoint.pt`  
Then run locally: `cd approaches/02_MultiTaskCNN && python run.py`

In [ ]:
# Cell 1: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Clone repo
import os

REPO      = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

APPROACH_DIR = f'{REPO_PATH}/approaches/02_MultiTaskCNN'
print('Approach dir:', APPROACH_DIR)

In [ ]:
# Cell 4: Configure training
EPOCHS = 100   # 50-100 is enough for good accuracy

print(f'Will train for {EPOCHS} epochs')

In [ ]:
# Cell 5: Train
os.chdir(APPROACH_DIR)
!python run.py --train --epochs {EPOCHS}

In [ ]:
# Cell 6: Save checkpoint to Drive + download
import shutil
from google.colab import files

ckpt_src  = f'{APPROACH_DIR}/checkpoints/checkpoint.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints/02_MultiTaskCNN'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'checkpoint.pt'))
print('Saved to Drive:', drive_dir)

files.download(ckpt_src)
print('\nDownload started.')
print('Place checkpoint.pt in:  approaches/02_MultiTaskCNN/checkpoints/')
print('Then run locally:        cd approaches/02_MultiTaskCNN && python run.py')

In [ ]:
# Cell 7: Preview — show character accuracy per class
import sys
sys.path.insert(0, APPROACH_DIR)

import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torchvision.transforms as T

from dataset import load_all_writers, CHAR_TO_LABEL
from model   import MultiTaskCNN

LABEL_TO_CHAR = {v: k for k, v in CHAR_TO_LABEL.items()}
_tf = T.Compose([T.Grayscale(1), T.Resize((128,128)), T.ToTensor(), T.Normalize((0.5,),(0.5,))])

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt    = torch.load(f'{APPROACH_DIR}/checkpoints/checkpoint.pt', map_location=device)
model   = MultiTaskCNN(num_writers=ckpt['num_writers']).to(device)
model.load_state_dict(ckpt['model']); model.eval()

DATA_ROOT = f'{REPO_PATH}/Data/Writers_pngs'
data      = load_all_writers(DATA_ROOT)

# Show first 10 characters with predictions
samples = data[:10]
fig, axes = plt.subplots(1, 10, figsize=(20, 3))
for ax, (path, true_label) in zip(axes, samples):
    img = Image.open(path).convert('L')
    with torch.no_grad():
        char_out, _ = model(_tf(img).unsqueeze(0).to(device))
    pred = LABEL_TO_CHAR.get(char_out.argmax(1).item(), '?')
    true = LABEL_TO_CHAR.get(true_label, '?')
    ax.imshow(img, cmap='gray')
    color = 'green' if pred == true else 'red'
    ax.set_title(f'T:{true}\nP:{pred}', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle(f'MultiTaskCNN predictions after {EPOCHS} epochs  (green=correct)')
plt.tight_layout()
plt.show()